# Tech Challenge — Fase 2
## Notebook 00 — Explorador do Data Lake

Este notebook não faz parte da pipeline: é a ferramenta para **olhar** o que ela produziu.
As camadas Bronze, Silver e Gold ficam em Parquet dentro de `data/lake/`, que é um formato binário —
não dá para abrir no Excel nem ler com um editor de texto.

Aqui você consegue:

1. listar todas as tabelas das três camadas, com linhas, colunas e tamanho;
2. espiar qualquer tabela com `ver("nome_da_tabela")`;
3. rodar SQL sobre qualquer tabela do lake com `consultar("SELECT ...")`;
4. exportar para CSV/Excel, se quiser abrir fora do notebook.

Rode os Notebooks 01 a 04 antes deste.

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install pyarrow duckdb --quiet

import os
import sys
from pathlib import Path

import pandas as pd
import duckdb

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)


def localizar_raiz_projeto():
    for candidato in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidato / "data" / "landing").is_dir():
            return candidato
    raise RuntimeError("Abra este notebook de dentro do repositório.")


PROJECT_ROOT = localizar_raiz_projeto()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.lakeio import ler_parquet

LAKE = Path(os.environ.get("TC2_LAKE", "data/lake"))
CAMADAS = {
    "bronze": LAKE / "bronze",
    "bronze_streaming": LAKE / "bronze_streaming",
    "silver": LAKE / "silver",
    "gold": LAKE / "gold",
}

print(f"Data lake: {LAKE.resolve()}")
for nome, caminho in CAMADAS.items():
    existe = "ok" if caminho.exists() else "AUSENTE (rode os notebooks da pipeline)"
    print(f"  {nome:18s} {existe}")

In [ ]:
# ============================================================
# CATÁLOGO: TODAS AS TABELAS DO LAKE
# ============================================================
def tamanho(caminho):
    caminho = Path(caminho)
    if caminho.is_file():
        return caminho.stat().st_size
    return sum(f.stat().st_size for f in caminho.rglob("*") if f.is_file())


def formatar_bytes(n):
    n = float(n)
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:.1f} {unidade}"
        n /= 1024


TABELAS = {}      # nome -> caminho
linhas_catalogo = []

for camada, raiz in CAMADAS.items():
    if not raiz.exists():
        continue
    for pasta in sorted(raiz.iterdir()):
        # ignora os arquivos de controle (_manifesto_*.json, _inventario_*.json)
        if not pasta.is_dir() or pasta.name.startswith("_"):
            continue
        try:
            df = ler_parquet(pasta)
        except Exception as exc:
            print(f"[aviso] não consegui ler {pasta}: {exc}")
            continue

        TABELAS[pasta.name] = pasta
        particoes = sorted({p.name.split("=")[0] for p in pasta.rglob("*=*") if p.is_dir()})
        linhas_catalogo.append({
            "camada": camada,
            "tabela": pasta.name,
            "linhas": len(df),
            "colunas": len(df.columns),
            "particionada_por": ", ".join(particoes) if particoes else "-",
            "tamanho": formatar_bytes(tamanho(pasta)),
        })

catalogo = pd.DataFrame(linhas_catalogo)
print(f"{len(catalogo)} tabelas | {catalogo['linhas'].sum():,} linhas no total\n".replace(",", "."))
display(catalogo)

---
## Espiando uma tabela

`ver("nome_da_tabela")` mostra o schema, algumas linhas e as estatísticas básicas. O nome é o mesmo
da coluna `tabela` do catálogo acima — por exemplo `fato_alfabetizacao_municipio`,
`gold_indicador_municipio` ou `indicador_municipio`.

In [ ]:
# ============================================================
# VER UMA TABELA
# ============================================================
def ver(tabela, linhas=8, estatisticas=True):
    """Mostra schema, amostra e estatísticas de uma tabela do lake."""
    if tabela not in TABELAS:
        print(f"Tabela '{tabela}' não encontrada. Disponíveis:")
        print("  " + "\n  ".join(sorted(TABELAS)))
        return None

    df = ler_parquet(TABELAS[tabela])
    print("=" * 90)
    print(f"  {tabela}   |   {len(df):,} linhas x {len(df.columns)} colunas".replace(",", "."))
    print("=" * 90)

    schema = pd.DataFrame({
        "coluna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "nulos_%": [round(df[c].isna().mean() * 100, 1) for c in df.columns],
        "exemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    print("\nSCHEMA")
    display(schema)

    print(f"\nPRIMEIRAS {linhas} LINHAS")
    display(df.head(linhas))

    if estatisticas:
        numericas = df.select_dtypes("number")
        if not numericas.empty:
            print("\nESTATÍSTICAS (colunas numéricas)")
            display(numericas.describe().T.round(2))
    return df


# Exemplo: a tabela central da camada Silver
_ = ver("fato_alfabetizacao_municipio")

---
## SQL sobre o lake

Todas as tabelas ficam registradas no DuckDB com o próprio nome, então dá para consultar qualquer
uma com SQL — inclusive fazendo join entre camadas. É a mesma coisa que o Athena faria sobre o S3,
só que rodando na sua máquina.

In [ ]:
# ============================================================
# CONSULTAR O LAKE COM SQL
# ============================================================
conexao = duckdb.connect()
for nome, caminho in TABELAS.items():
    conexao.register(nome, ler_parquet(caminho))


def consultar(sql):
    """Roda SQL sobre qualquer tabela do lake."""
    return conexao.execute(sql).df()


print(f"{len(TABELAS)} tabelas registradas no DuckDB.\n")

print("Exemplo 1 — piores municípios da rede municipal no ciclo mais recente:")
display(consultar("""
    SELECT nome_municipio, sigla_uf, taxa_alfabetizacao, meta_taxa_ano, gap_meta_2030
    FROM gold_indicador_municipio
    WHERE ano = 2024 AND rede_nome = 'Municipal'
    ORDER BY taxa_alfabetizacao
    LIMIT 10
"""))

print("\nExemplo 2 — join entre Gold e a tabela de priorização:")
display(consultar("""
    SELECT p.sigla_uf,
           COUNT(*) AS municipios_criticos,
           ROUND(AVG(p.taxa_alfabetizacao), 1) AS taxa_media,
           ROUND(AVG(g.ritmo_anual_necessario_pp), 2) AS ritmo_necessario_pp_ano
    FROM gold_priorizacao_municipio p
    JOIN gold_indicador_municipio g
      ON g.id_municipio = p.id_municipio AND g.ano = 2024 AND g.rede_nome = 'Municipal'
    WHERE p.faixa_prioridade = 'Crítica'
    GROUP BY p.sigla_uf
    ORDER BY municipios_criticos DESC
    LIMIT 10
"""))

---
## Exportar para CSV

Se quiser abrir as tabelas no Excel ou mandar para alguém, a célula abaixo exporta as principais
para `data/export/`. Essa pasta fica fora do Git (o Parquet no lake continua sendo a fonte da
verdade; o CSV é só para visualização).

In [ ]:
# ============================================================
# EXPORTAR PARA CSV
# ============================================================
EXPORTAR = [
    "fato_alfabetizacao_municipio",
    "gold_indicador_municipio",
    "gold_meta_vs_realizado",
    "gold_evolucao_temporal",
    "gold_priorizacao_municipio",
]

destino = Path("data/export")
destino.mkdir(parents=True, exist_ok=True)

for tabela in EXPORTAR:
    if tabela not in TABELAS:
        continue
    df = ler_parquet(TABELAS[tabela])
    arquivo = destino / f"{tabela}.csv"
    # sep=';' e decimal=',' para o Excel em português abrir sem bagunçar as colunas
    df.to_csv(arquivo, index=False, sep=";", decimal=",", encoding="utf-8-sig")
    print(f"{tabela:34s} -> {arquivo}  ({len(df):,} linhas)".replace(",", "."))

print(f"\nArquivos em: {destino.resolve()}")